# Test bigquery

Upload de catalogue pizeo

In [12]:
import os
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

In [13]:
# 1 Connexion
# 1.1 Charge les variables du fichier .env
load_dotenv(dotenv_path="../.env")

# 1.2. Récupère la valeur de la variable d'environnement
GCP_PROJECT_ID = os.environ.get("GCP_PROJECT_ID")

# 1.3. Initialise le client avec la variable définie
client = bigquery.Client(project=GCP_PROJECT_ID)

print(f"Connecté à {GCP_PROJECT_ID} ✅")

Connecté à hydro-sense-498112 ✅


In [14]:
print(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))
print(os.getenv("GCP_PROJECT_ID"))

/home/charourou/code/charourou/gcp/lewagonds-3b7959ddc154.json
hydro-sense-498112


## 2. Load test model "piezo_test"

In [15]:
# 2 Load test model "piezo_test"
DATASET = os.environ.get("BQ_DATASET_ID")
TABLE = "piezo_test"

query = f"""
    SELECT *
    FROM {GCP_PROJECT_ID}.{DATASET}.{TABLE}
    LIMIT 100
"""

query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()
df.head()

Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/hydro-sense-498112/jobs?prettyPrint=false: Access Denied: Project hydro-sense-498112: User does not have bigquery.jobs.create permission in project hydro-sense-498112.

Location: None
Job ID: 472dc8bc-a54d-4c63-ba96-c652c6fd8392


In [ ]:
# problem of parsing ?? régler plus tard ??

In [ ]:
# 3 test uploadinf

df_test = pd.read_csv(f'../raw_data/piezo_BSS001BLTP.csv', sep = ';')
df_test.head()

,date_mesure,niveau_nappe_eau,profondeur_nappe
0,1994-04-22,44.19,44.19
1,1994-04-23,44.29,44.29
2,1994-04-24,44.26,44.26
3,1994-04-25,44.22,44.22
4,1994-04-26,44.19,44.19


In [11]:
TABLE = "piezo_upload_test"

table = f"{GCP_PROJECT_ID}.{DATASET}.{TABLE}"

client = bigquery.Client()

write_mode = "WRITE_TRUNCATE" # or "WRITE_APPEND"
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(df_test, table, job_config=job_config)
result = job.result()

/home/yanns/.pyenv/versions/3.10.6/envs/Projet_Hydrosense/lib/python3.10/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


In [14]:
TABLE = "piezo_upload_test"

query = f"""
    SELECT *
    FROM {GCP_PROJECT_ID}.{DATASET}.{TABLE}
    LIMIT 100
"""

query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()
df.head()

/home/yanns/.pyenv/versions/3.10.6/envs/Projet_Hydrosense/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_mesure,niveau_nappe_eau,profondeur_nappe
0,2005-10-13,36.88,36.88
1,2005-10-12,36.90,36.90
2,2005-10-09,36.92,36.92
3,2005-10-11,36.92,36.92
4,2005-10-10,36.94,36.94


## 4. Upload de catalogue piezo

Catalogue des métadonnées des stations du réseau Ades

In [17]:
from hydrosense.database.ades import GestionnairePiezometrie
from hydrosense.database.entite import Entite